# Random Survival Forests
22조 홍태선

In [1]:
!pip install -q lifelines scikit-survival

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from lifelines.utils import concordance_index

from itertools import product

from sklearn.model_selection import StratifiedKFold
from sklearn.inspection import permutation_importance

pd.set_option("display.max_columns", None)
print("imports done")

imports done


In [2]:
HORIZONS  = [12, 24, 48, 72]
N_SPLITS  = 5
SEED      = 42
W_CINDEX  = 0.3
W_BRIER   = 0.7

TRAIN_PATH = "train.csv"
TEST_PATH  = "test.csv"
SUB_PATH   = "sample_submission.csv"

ID_COL    = "event_id"
EVENT_COL = "event"
TIME_COL  = "time_to_hit_hours"

print("constants set")

constants set


In [3]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sub   = pd.read_csv(SUB_PATH)
print("train:", train.shape, "  test:", test.shape)
print("columns:", train.columns.tolist())
train.head(3)

train: (221, 37)   test: (95, 35)
columns: ['event_id', 'num_perimeters_0_5h', 'dt_first_last_0_5h', 'low_temporal_resolution_0_5h', 'area_first_ha', 'area_growth_abs_0_5h', 'area_growth_rel_0_5h', 'area_growth_rate_ha_per_h', 'log1p_area_first', 'log1p_growth', 'log_area_ratio_0_5h', 'relative_growth_0_5h', 'radial_growth_m', 'radial_growth_rate_m_per_h', 'centroid_displacement_m', 'centroid_speed_m_per_h', 'spread_bearing_deg', 'spread_bearing_sin', 'spread_bearing_cos', 'dist_min_ci_0_5h', 'dist_std_ci_0_5h', 'dist_change_ci_0_5h', 'dist_slope_ci_0_5h', 'closing_speed_m_per_h', 'closing_speed_abs_m_per_h', 'projected_advance_m', 'dist_accel_m_per_h2', 'dist_fit_r2_0_5h', 'alignment_cos', 'alignment_abs', 'cross_track_component', 'along_track_speed', 'event_start_hour', 'event_start_dayofweek', 'event_start_month', 'time_to_hit_hours', 'event']


,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,log_area_ratio_0_5h,relative_growth_0_5h,radial_growth_m,radial_growth_rate_m_per_h,centroid_displacement_m,centroid_speed_m_per_h,spread_bearing_deg,spread_bearing_sin,spread_bearing_cos,dist_min_ci_0_5h,dist_std_ci_0_5h,dist_change_ci_0_5h,dist_slope_ci_0_5h,closing_speed_m_per_h,closing_speed_abs_m_per_h,projected_advance_m,dist_accel_m_per_h2,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,10892457,3,4.265188,0,79.696304,2.875935,0.036086,0.674281,4.390693,1.354787,0.03545,0.036086,9.007182,2.11179,8.274971,1.940119,70.130507,0.940469,0.339879,6166.121596,0.205085,0.435052,1.090997e-01,-0.102001,0.102001,-0.435052,7.275611e-02,0.886373,-0.054649,0.054649,-1.937219,-0.106026,19,4,5,18.892512,0
1,11757157,2,1.169918,0,8.946749,0.000000,0.000000,0.000000,2.297246,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,1.000000,2930.925956,0.000000,0.000000,-3.887003e-13,0.000000,0.000000,0.000000,0.000000e+00,0.000000,-0.568898,0.568898,-0.000000,-0.000000,4,4,6,22.048108,1
2,11945086,4,4.777526,0,106.482638,0.000000,0.000000,0.000000,4.677329,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,1.000000,3272.375090,0.000000,0.000000,-1.390327e-13,0.000000,0.000000,0.000000,7.965118e-14,0.000000,0.882385,0.882385,0.000000,0.000000,22,4,8,0.888895,1


In [4]:
def add_features(df):
    df = df.copy()
    # log1p 변환
    df["log1p_dist_min_ci_0_5h"] = np.log1p(df["dist_min_ci_0_5h"])
    return df

train = add_features(train)
test  = add_features(test)
print("feature engineering done")
print("train shape:", train.shape)

feature engineering done
train shape: (221, 38)


In [5]:
def fit_preprocess_rsf(X_tr_raw, X_va_raw=None, X_te_raw=None, use_corr_filter=False, corr_threshold=0.98):
    """
    RSF 전용 전처리
    - median imputation
    - 저분산 변수 제거
    - 필요할 때만 고상관 변수 제거
    - StandardScaler 없음
    """
    X_tr = X_tr_raw.copy()

    medians = X_tr.median(numeric_only=True)
    X_tr = X_tr.fillna(medians)

    to_drop = set()
    if use_corr_filter:
        corr_matrix = X_tr.corr(numeric_only=True).abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        protected_cols = {"log1p_dist_min_ci_0_5h"}

        for col in upper.columns:
            if col in protected_cols:
                continue
            if any(upper[col] > corr_threshold):
                to_drop.add(col)

        if len(to_drop) > 0:
            X_tr = X_tr.drop(columns=list(to_drop), errors="ignore")

    low_var = X_tr.columns[X_tr.var(numeric_only=True) < 1e-6].tolist()
    X_tr = X_tr.drop(columns=low_var, errors="ignore")

    feature_cols = X_tr.columns.tolist()

    def transform(X_raw):
        X = X_raw.copy()
        X = X.fillna(medians)
        X = X.drop(columns=list(to_drop), errors="ignore")
        X = X.drop(columns=low_var, errors="ignore")
        return X[feature_cols]

    result = {
        "feature_cols": feature_cols,
        "medians": medians,
        "dropped_corr": list(to_drop),
        "dropped_var": low_var,
        "X_tr": X_tr
    }

    if X_va_raw is not None:
        result["X_va"] = transform(X_va_raw)
    if X_te_raw is not None:
        result["X_te"] = transform(X_te_raw)

    return result

print("fit_preprocess_rsf defined")

fit_preprocess_rsf defined


In [6]:
def brier_score_censored_comp(times, events, pred_event_prob, H):
    """
    baseline과 동일한 방식
    - event=1 and time<=H  -> label=1, include
    - event=1 and time>H   -> label=0, include
    - event=0 and time>=H  -> label=0, include
    - event=0 and time<H   -> censored before horizon, exclude
    """
    times = np.asarray(times)
    events = np.asarray(events)
    pred_event_prob = np.asarray(pred_event_prob)

    include_mask = (events == 1) | ((events == 0) & (times >= H))
    y_true = ((events == 1) & (times <= H)).astype(float)

    y_true = y_true[include_mask]
    y_pred = pred_event_prob[include_mask]

    if len(y_true) == 0:
        return np.nan, 0

    bs = np.mean((y_true - y_pred) ** 2)
    return bs, len(y_true)

print("Brier function defined")

Brier function defined


In [7]:
def compute_cindex_baseline_style(time, event, risk_score):
    time = np.asarray(time)
    event = np.asarray(event)
    risk_score = np.asarray(risk_score)

    return concordance_index(time, -risk_score, event)

print("c-index function defined")

c-index function defined


In [8]:
def run_rsf_cv(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=5,
    max_features=None,
    min_samples_split=2,
    bootstrap=True,
    use_corr_filter=False
):
    """RSF 5-Fold 교차검증 (Lasso-Cox와 동일 metric)"""

    exclude_cols = [ID_COL, EVENT_COL, TIME_COL, "dist_min_ci_0_5h"]
    raw_feature_cols = [
        c for c in train.columns
        if c not in exclude_cols and train[c].dtype != object
    ]

    X_all = train[raw_feature_cols].copy()
    y_event = train[EVENT_COL].values.astype(int)
    y_time = train[TIME_COL].values.astype(float)

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    oof_prob = np.zeros((len(train), len(HORIZONS)))
    oof_risk = np.zeros(len(train))
    fold_cindex = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y_event), start=1):
        X_tr_raw = X_all.iloc[tr_idx]
        X_va_raw = X_all.iloc[va_idx]

        prep = fit_preprocess_rsf(
            X_tr_raw,
            X_va_raw=X_va_raw,
            use_corr_filter=use_corr_filter
        )
        X_tr = prep["X_tr"]
        X_va = prep["X_va"]

        y_tr = Surv.from_arrays(
            event=y_event[tr_idx].astype(bool),
            time=y_time[tr_idx]
        )

        rsf = RandomSurvivalForest(
            n_estimators=int(n_estimators),
            max_depth=None if max_depth is None else int(max_depth),
            min_samples_leaf=int(min_samples_leaf),
            min_samples_split=int(min_samples_split),
            max_features=max_features,
            bootstrap=bootstrap,
            n_jobs=-1,
            random_state=SEED
        )
        rsf.fit(X_tr.values, y_tr)

        # risk score
        va_risk = rsf.predict(X_va.values)
        oof_risk[va_idx] = va_risk

        fold_cidx = compute_cindex_baseline_style(
            y_time[va_idx],
            y_event[va_idx],
            va_risk
        )
        fold_cindex.append(fold_cidx)

        # horizon별 OOF probability
        surv_funcs = rsf.predict_survival_function(X_va.values)
        for local_i, sf in enumerate(surv_funcs):
            for h_i, H in enumerate(HORIZONS):
                t = min(H, sf.x.max())
                oof_prob[va_idx[local_i], h_i] = np.clip(1.0 - sf(t), 0.0, 1.0)

        # monotonicity 보정
        oof_prob[va_idx] = np.maximum.accumulate(oof_prob[va_idx], axis=1)

    # OOF 전체 평가
    oof_cindex = compute_cindex_baseline_style(y_time, y_event, oof_risk)

    # 공식 metric:
    brier_24, n24 = brier_score_censored_comp(y_time, y_event, oof_prob[:, 1], 24)
    brier_48, n48 = brier_score_censored_comp(y_time, y_event, oof_prob[:, 2], 48)
    brier_72, n72 = brier_score_censored_comp(y_time, y_event, oof_prob[:, 3], 72)

    weighted_brier = 0.3 * brier_24 + 0.4 * brier_48 + 0.3 * brier_72
    hybrid_score = 0.3 * oof_cindex + 0.7 * (1 - weighted_brier)

    return {
        "fold_cindex_mean": float(np.mean(fold_cindex)),
        "fold_cindex_std": float(np.std(fold_cindex)),
        "oof_cindex": float(oof_cindex),
        "brier_24": float(brier_24),
        "brier_48": float(brier_48),
        "brier_72": float(brier_72),
        "weighted_brier": float(weighted_brier),
        "hybrid_score": float(hybrid_score),
        "n24": int(n24),
        "n48": int(n48),
        "n72": int(n72),
        "oof_prob": oof_prob,
        "oof_risk": oof_risk
    }

In [9]:
# ── Integrated grid search ─────────────────

print("=" * 60)
print("Integrated grid search")
print("=" * 60)

param_grid = { 
    "n_estimators": [470,480,490],
    "max_depth": [None],
    "min_samples_leaf": [3, 5],
    "max_features": [0.5, 0.8, "sqrt"],
    "min_samples_split": [2]
}

all_combinations = list(product(
    param_grid["n_estimators"],
    param_grid["max_depth"],
    param_grid["min_samples_leaf"],
    param_grid["max_features"],
    param_grid["min_samples_split"]
))

total_runs = len(all_combinations)
results_final = []

for i, (n_est, depth, leaf, mf, mss) in enumerate(all_combinations, start=1):
    res = run_rsf_cv(
        n_estimators=n_est,
        max_depth=depth,
        min_samples_leaf=leaf,
        max_features=mf,
        min_samples_split=mss,
        bootstrap=True,
        use_corr_filter=False
    )

    results_final.append({
        "n_estimators": n_est,
        "max_depth": depth,
        "min_samples_leaf": leaf,
        "max_features": mf,
        "min_samples_split": mss,
        "fold_cindex_mean": res["fold_cindex_mean"],
        "fold_cindex_std": res["fold_cindex_std"],
        "oof_cindex": res["oof_cindex"],
        "brier_24": res["brier_24"],
        "brier_48": res["brier_48"],
        "brier_72": res["brier_72"],
        "weighted_brier": res["weighted_brier"],
        "hybrid_score": res["hybrid_score"]
    })

    if i % 5 == 0 or i == total_runs:
        print(f"Progress: {i}/{total_runs} ({i/total_runs*100:.1f}%)")

df_final = pd.DataFrame(results_final).sort_values("hybrid_score", ascending=False)

print("\n── grid 결과 상위 20개 ──")
print(df_final.head(20).to_string(index=False))

Integrated grid search
Progress: 5/18 (27.8%)
Progress: 10/18 (55.6%)
Progress: 15/18 (83.3%)
Progress: 18/18 (100.0%)

── grid 결과 상위 20개 ──
 n_estimators max_depth  min_samples_leaf max_features  min_samples_split  fold_cindex_mean  fold_cindex_std  oof_cindex  brier_24  brier_48  brier_72  weighted_brier  hybrid_score
          480      None                 3          0.8                  2          0.950447         0.009456    0.942282  0.027704  0.014962  0.000118        0.014331      0.972653
          470      None                 3          0.8                  2          0.950447         0.009456    0.942117  0.027705  0.014993  0.000123        0.014346      0.972593
          490      None                 3          0.8                  2          0.950020         0.009182    0.941868  0.027734  0.014971  0.000133        0.014348      0.972517
          480      None                 5          0.8                  2          0.948836         0.006578    0.942613  0.028174  0.0

In [10]:
# ── 최적 파라미터 확정 ────────────────────────────────────────
best_final = df_final.iloc[0]

BEST_N_EST = int(best_final["n_estimators"])
BEST_DEPTH = best_final["max_depth"]
BEST_LEAF = int(best_final["min_samples_leaf"])
BEST_MAX_FEATURES = best_final["max_features"]
BEST_MIN_SAMPLES_SPLIT = int(best_final["min_samples_split"])
BEST_BOOTSTRAP = True

print("=" * 40)
print("최종 RSF 파라미터")
print(f"  n_estimators      : {BEST_N_EST}")
print(f"  max_depth         : {BEST_DEPTH}")
print(f"  min_samples_leaf  : {BEST_LEAF}")
print(f"  max_features      : {BEST_MAX_FEATURES}")
print(f"  min_samples_split : {BEST_MIN_SAMPLES_SPLIT}")
print(f"  Hybrid Score      : {best_final['hybrid_score']:.6f}")
print("=" * 40)

최종 RSF 파라미터
  n_estimators      : 480
  max_depth         : None
  min_samples_leaf  : 3
  max_features      : 0.8
  min_samples_split : 2
  Hybrid Score      : 0.972653


In [11]:
# ── 전체 학습 데이터로 최종 모델 피팅 ────────────────────────
exclude_cols = [ID_COL, EVENT_COL, TIME_COL, "dist_min_ci_0_5h"]
raw_feature_cols = [
    c for c in train.columns
    if c not in exclude_cols and train[c].dtype != object
]

X_all = train[raw_feature_cols].copy()
X_test = test[raw_feature_cols].copy()

max_features=BEST_MAX_FEATURES,

full_prep = fit_preprocess_rsf(
    X_all,
    X_te_raw=X_test,
    use_corr_filter=False
)

X_full = full_prep["X_tr"]
X_te = full_prep["X_te"]

y_full = Surv.from_arrays(
    event=train[EVENT_COL].values.astype(bool),
    time=train[TIME_COL].values.astype(float)
)

final_rsf = RandomSurvivalForest(
    n_estimators=int(BEST_N_EST),
    max_depth=None if pd.isna(BEST_DEPTH) else int(BEST_DEPTH),
    min_samples_leaf=int(BEST_LEAF),
    max_features=BEST_MAX_FEATURES,
    min_samples_split=int(BEST_MIN_SAMPLES_SPLIT),
    bootstrap=BEST_BOOTSTRAP,
    n_jobs=-1,
    random_state=SEED
)

final_rsf.fit(X_full.values, y_full)
print("Final RSF fitted")
print("n_features:", X_full.shape[1])

Final RSF fitted
n_features: 34


In [12]:
# # 피처 중요도 계산 (Permutation Importance)
# perm = permutation_importance(
#     final_rsf, X_full.values, y_full,
#     n_repeats=5, random_state=SEED, n_jobs=-1
# )
# fi_values = perm.importances_mean

# fi = pd.DataFrame({
#     "feature":    full_prep["feature_cols"],
#     "importance": fi_values
# }).sort_values("importance", ascending=False)

# print(fi.head(15).to_string(index=False))

# plt.figure(figsize=(8, 6))
# plt.barh(fi["feature"][:15][::-1], fi["importance"][:15][::-1])
# plt.xlabel("Permutation Importance")
# plt.title("RSF Feature Importance (Top 15)")
# plt.tight_layout()
# plt.show()

In [13]:
# ── 테스트 예측 및 단조성 보정 ────────────────────────────────
surv_te = final_rsf.predict_survival_function(X_te.values)
test_risk = final_rsf.predict(X_te.values)

n_test = len(test)
test_probs = np.zeros((n_test, len(HORIZONS)))

for i, sf in enumerate(surv_te):
    for h_i, H in enumerate(HORIZONS):
        t = min(H, sf.x.max())
        test_probs[i, h_i] = np.clip(1.0 - sf(t), 0.0, 1.0)

# 단조성 보정
test_probs = np.maximum.accumulate(test_probs, axis=1)

print("test prediction done", test_probs.shape)
print(pd.DataFrame(test_probs, columns=[f"H{h}" for h in HORIZONS]).describe())

test prediction done (95, 4)
             H12        H24        H48        H72
count  95.000000  95.000000  95.000000  95.000000
mean    0.185863   0.266129   0.284355   0.298617
std     0.300856   0.403671   0.430379   0.451044
min     0.000000   0.000000   0.000000   0.000000
25%     0.000000   0.000000   0.000000   0.000000
50%     0.000000   0.000000   0.000000   0.000000
75%     0.381885   0.795839   0.882974   0.996363
max     0.988672   0.998661   1.000000   1.000000


In [14]:
# ── 제출 파일 생성 ────────────────────────────────────────────
horizon_cols = [c for c in sub.columns if c != ID_COL]
assert len(horizon_cols) == len(HORIZONS), f"horizon mismatch: {horizon_cols}"
print("submission columns:", horizon_cols)
print("model horizons     :", HORIZONS)

sub_out = sub.copy()
for h_i, col in enumerate(horizon_cols):
    sub_out[col] = test_probs[:, h_i]

sub_out.to_csv("rsf_submission.csv", index=False)
print("Saved → rsf_submission.csv")
sub_out.head()

submission columns: ['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']
model horizons     : [12, 24, 48, 72]
Saved → rsf_submission.csv


,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.000000,0.000000,0.000000,0.000000
1,13353600,0.509709,0.959105,0.999508,1.000000
2,13942327,0.000000,0.000000,0.000000,0.000000
3,16112781,0.769234,0.967120,0.990738,1.000000
4,17132808,0.150799,0.150799,0.150799,0.150799


In [15]:
best_res = run_rsf_cv(
    n_estimators=BEST_N_EST,
    max_depth=BEST_DEPTH,
    min_samples_leaf=BEST_LEAF,
    max_features=BEST_MAX_FEATURES,
    min_samples_split=BEST_MIN_SAMPLES_SPLIT,
    bootstrap=BEST_BOOTSTRAP,
    use_corr_filter=False
)

np.save("rsf_oof_prob.npy", best_res["oof_prob"])
np.save("rsf_oof_risk.npy", best_res["oof_risk"])